# MG Outliers Detection Report

Rapport de détection des valeurs aberrantes **Magic Glasses** (MG) sur les données DHIS2 formatées.

- **Méthode partielle** : MAD15 → MAD10 (outliers partiels).
- **Méthode complète** : MAD15 → MAD10 → seasonal5 → seasonal3 (outliers complets).

Les sections ci-dessous résument le nombre d’outliers détectés et leur répartition par indicateur et par année.

In [ ]:
# Setup
ROOT_PATH <- "~/workspace"
PIPELINE_PATH <- file.path(ROOT_PATH, "pipelines", "snt_dhis2_outliers_imputation_magic_glasses")

source(file.path(PIPELINE_PATH, "utils", "bootstrap.R"))
setup_ctx <- bootstrap_magic_glasses_context(
  root_path = ROOT_PATH,
  required_packages = c("jsonlite", "arrow", "glue", "reticulate", "dplyr", "ggplot2", "knitr", "scales")
)

CODE_PATH <- setup_ctx$CODE_PATH
CONFIG_PATH <- setup_ctx$CONFIG_PATH
DATA_PATH <- file.path(setup_ctx$DATA_PATH, "dhis2", "outliers_imputation")
openhexa <- setup_ctx$openhexa

config_json <- fromJSON(file.path(CONFIG_PATH, "SNT_config.json"))
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
COUNTRY_NAME <- config_json$SNT_CONFIG$COUNTRY_NAME

detected_fp <- file.path(DATA_PATH, paste0(COUNTRY_CODE, "_routine_outliers_detected.parquet"))

detected_tbl <- NULL
partial_tbl <- NULL
complete_tbl <- NULL
if (file.exists(detected_fp)) {
  detected_tbl <- as.data.frame(arrow::read_parquet(detected_fp))

  if (!"OUTLIER_DETECTED" %in% names(detected_tbl)) {
    legacy_outlier_cols <- grep("^OUTLIER_", names(detected_tbl), value = TRUE)
    if (length(legacy_outlier_cols) > 0) {
      detected_tbl$OUTLIER_DETECTED <- detected_tbl[[legacy_outlier_cols[1]]]
    } else {
      detected_tbl$OUTLIER_DETECTED <- FALSE
    }
  }

  if (!"OUTLIER_METHOD" %in% names(detected_tbl)) {
    detected_tbl$OUTLIER_METHOD <- "MAGIC_GLASSES_PARTIAL"
  }

  detected_tbl <- detected_tbl %>% filter(OUTLIER_DETECTED == TRUE)
  log_msg(glue("MG detected outliers: {nrow(detected_tbl)} rows"))

  partial_tbl <- detected_tbl %>% filter(OUTLIER_METHOD == "MAGIC_GLASSES_PARTIAL")
  complete_tbl <- detected_tbl %>% filter(OUTLIER_METHOD == "MAGIC_GLASSES_COMPLETE")

  log_msg(glue("MG partial outliers: {nrow(partial_tbl)} rows"))
  log_msg(glue("MG complete outliers: {nrow(complete_tbl)} rows"))
} else {
  log_msg("MG detected outlier file not found.")
}

## 1. Résumé des outliers détectés

In [ ]:
# Tableau récapitulatif (totaux)
summary_totals <- data.frame(
  Méthode = c("MG partiel (MAD15 → MAD10)", "MG complet (+ seasonal5 → seasonal3)"),
  Nombre_d_outliers = c(
    if (!is.null(partial_tbl)) nrow(partial_tbl) else 0L,
    if (!is.null(complete_tbl)) nrow(complete_tbl) else 0L
  )
)
knitr::kable(summary_totals, format = "simple", col.names = c("Méthode", "Nombre d'outliers"))

## 2. Répartition par indicateur

In [ ]:
# Nombre d'outliers par indicateur (méthode partielle)
if (!is.null(partial_tbl) && nrow(partial_tbl) > 0 && "INDICATOR" %in% names(partial_tbl)) {
  by_ind_partial <- partial_tbl %>%
    group_by(INDICATOR) %>%
    summarise(n_outliers = n(), .groups = "drop") %>%
    arrange(desc(n_outliers))
  knitr::kable(by_ind_partial, format = "simple", col.names = c("Indicateur", "Nombre d'outliers"))
} else {
  cat("Aucune donnée partielle disponible.\n")
}

In [ ]:
# Nombre d'outliers par indicateur (méthode complète)
if (!is.null(complete_tbl) && nrow(complete_tbl) > 0 && "INDICATOR" %in% names(complete_tbl)) {
  by_ind_complete <- complete_tbl %>%
    group_by(INDICATOR) %>%
    summarise(n_outliers = n(), .groups = "drop") %>%
    arrange(desc(n_outliers))
  knitr::kable(by_ind_complete, format = "simple", col.names = c("Indicateur", "Nombre d'outliers"))
} else {
  cat("Aucune donnée complète disponible (ou méthode complète non exécutée).\n")
}

In [ ]:
# Graphique : répartition par indicateur (partiel et/ou complet)
plot_df <- NULL
if (!is.null(partial_tbl) && nrow(partial_tbl) > 0 && "INDICATOR" %in% names(partial_tbl)) {
  plot_df <- partial_tbl %>%
    group_by(INDICATOR) %>%
    summarise(n = n(), .groups = "drop") %>%
    mutate(méthode = "Partiel")
}
if (!is.null(complete_tbl) && nrow(complete_tbl) > 0 && "INDICATOR" %in% names(complete_tbl)) {
  comp <- complete_tbl %>%
    group_by(INDICATOR) %>%
    summarise(n = n(), .groups = "drop") %>%
    mutate(méthode = "Complet")
  plot_df <- if (is.null(plot_df)) comp else bind_rows(plot_df, comp)
}
if (!is.null(plot_df) && nrow(plot_df) > 0) {
  n_methods <- n_distinct(plot_df$méthode)
  p <- ggplot(plot_df, aes(x = reorder(INDICATOR, n), y = n, fill = méthode)) +
    geom_col(position = if (n_methods > 1) "dodge" else "stack", alpha = 0.85) +
    coord_flip() +
    scale_y_continuous(labels = scales::label_number(big.mark = " ")) +
    labs(
      title = "Nombre d'outliers MG par indicateur",
      subtitle = paste0(COUNTRY_NAME, " (", COUNTRY_CODE, ")"),
      x = "Indicateur", y = "Nombre d'outliers", fill = "Méthode"
    ) +
    theme_minimal(base_size = 12) +
    theme(legend.position = "bottom")
  print(p)
} else {
  cat("Aucune donnée à afficher.\n")
}

## 3. Évolution par année

In [ ]:
# Tableau : nombre d'outliers par année (partiel et/ou complet)
year_tab <- NULL
if (!is.null(partial_tbl) && "YEAR" %in% names(partial_tbl)) {
  year_tab <- partial_tbl %>% group_by(YEAR) %>% summarise(n = n(), .groups = "drop") %>% mutate(méthode = "Partiel")
}
if (!is.null(complete_tbl) && "YEAR" %in% names(complete_tbl)) {
  yc <- complete_tbl %>% group_by(YEAR) %>% summarise(n = n(), .groups = "drop") %>% mutate(méthode = "Complet")
  year_tab <- if (is.null(year_tab)) yc else bind_rows(year_tab, yc)
}
if (!is.null(year_tab) && nrow(year_tab) > 0) {
  knitr::kable(year_tab, format = "simple", col.names = c("Année", "Nombre d'outliers", "Méthode"))
} else {
  cat("Aucune donnée par année disponible.\n")
}

In [ ]:
# Graphique : évolution du nombre d'outliers par année
year_df <- NULL
if (!is.null(partial_tbl) && nrow(partial_tbl) > 0 && "YEAR" %in% names(partial_tbl)) {
  year_df <- partial_tbl %>%
    group_by(YEAR) %>%
    summarise(n = n(), .groups = "drop") %>%
    mutate(méthode = "Partiel")
}
if (!is.null(complete_tbl) && nrow(complete_tbl) > 0 && "YEAR" %in% names(complete_tbl)) {
  yc <- complete_tbl %>%
    group_by(YEAR) %>%
    summarise(n = n(), .groups = "drop") %>%
    mutate(méthode = "Complet")
  year_df <- if (is.null(year_df)) yc else bind_rows(year_df, yc)
}
if (!is.null(year_df) && nrow(year_df) > 0) {
  p_year <- ggplot(year_df, aes(x = as.character(YEAR), y = n, fill = méthode)) +
    geom_col(position = "dodge", alpha = 0.85, width = 0.7) +
    scale_y_continuous(labels = scales::label_number(big.mark = " ")) +
    labs(
      title = "Évolution du nombre d'outliers MG par année",
      x = "Année", y = "Nombre d'outliers", fill = "Méthode"
    ) +
    theme_minimal(base_size = 12) +
    theme(legend.position = "bottom")
  print(p_year)
} else {
  cat("Aucune donnée par année à afficher.\n")
}